![image.png](https://i.imgur.com/a3uAqnb.png)
# Lab 9: World Models

This notebook implements ideas from **world models** — transition data collection, RSSM-style latent dynamics, and imagination rollouts.

You will gather CartPole transitions, build a miniature RSSM, and connect classic model-based RL to lecture topics (Dreamer, generative JEPA).

> 💡 A world model learns $p(s' \mid s, a)$ (or latent equivalents) for planning and sample-efficient control.

__Install dependencies, then proceed through theory and implementation.__



# 📦 Installing Required Python Libraries

This cell installs packages needed for this lab.

- **PyTorch** — RSSM modules and rollout loops.
- **Gymnasium** — CartPole environment for transition data.
- **Matplotlib / NumPy** — Visualizing imagined trajectories.


In [1]:
!pip install -q torch gymnasium matplotlib numpy scipy


# 📥 Importing Essential Python Libraries

Imports for environment interaction and RSSM training in Part B.


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
print(f"PyTorch {torch.__version__}")


PyTorch 2.11.0+cpu


__Theory first — connect definitions to the lecture slides, then move to code.__

---

## 🧠 Part A — Theory

*Reference: Ha & Schmidhuber VAE world model, RSSM, Dreamer, generative JEPA.*

### 📖 A1. Concepts

1. Formalize a world model: $s' \sim p(s' \mid s, a)$. What can $s$ contain?
2. Contrast **pixel prediction** vs **latent dynamics** for planning.
3. What is the role of the **world model** in Dreamer-style RL?

*Write below:*


#### ✍️ Your answers (A1):

1. $s$ may contain **proprioceptive state**, **observations** (pixels or features), **history**, or **latent summaries** sufficient to predict future evolution.

2. **Pixel prediction** is high-dimensional and noisy; **latent dynamics** focus on **compact state** useful for **planning**, improving **sample efficiency**.

3. In **Dreamer**, the world model enables **imagined rollouts** to train the policy **without environment interaction**, improving **sample efficiency**.

### 📖 A2. LeWorldModel

1. Why does LeWorldModel use multiple learning goals at once?
2. How does LeWorldModel learn good representations using contrastive learning?
3. How does LeWorldModel use two main loss terms to learn effectively?

*Write below:*

#### ✍️ Your answers (A2):

1. LeWorldModel uses many different learning goals (like understanding what it sees, predicting the future, and comparing different views) to help it learn versatile internal ideas (representations) that are good for many tasks. This helps it learn faster and apply what it learns more broadly.

2. LeWorldModel learns good representations by making sure that similar experiences (like an observation and what it becomes next) are close together in its internal understanding, while different experiences are far apart. This helps it create strong and clear internal representations.

3. LeWorldModel learns effectively using two simple loss terms: a prediction loss (to accurately forecast future states) and a regularizer that encourages its internal representations to be Gaussian-distributed. This simple, streamlined approach helps avoid complex loss functions and ensures stable and robust training.

---

## 💻 Part B — Programming
__Let's implement the core ideas in PyTorch.__

### 🛠️ B1. CartPole transition dataset

A world model learns $p(s' \mid s, a)$ from **transition tuples** $(s_t, a_t, s_{t+1}, r_t)$. Collecting rollouts from a simple environment is the first step before learning latent dynamics.

In [3]:
import gymnasium as gym
import torch.nn.functional as F

env = gym.make("CartPole-v1")
transitions = []
obs, _ = env.reset()
total = 0
while total < 500:
    action = env.action_space.sample()
    next_obs, reward, terminated, truncated, _ = env.step(action)
    transitions.append((obs.copy(), action, next_obs.copy(), reward))
    obs = next_obs
    total += 1
    if terminated or truncated:
        obs, _ = env.reset()
env.close()
print(f"Collected {len(transitions)} transitions.")
print(f"Sample transition: obs_dim={transitions[0][0].shape}, reward={transitions[0][3]}")

Collected 500 transitions.
Sample transition: obs_dim=(4,), reward=1.0


### 🛠️ B2. Tiny RSSM module

 The **Recurrent State-Space Model (RSSM)** combines a **deterministic** GRU state $h_t$ with a **stochastic** latent $z_t$. An encoder maps observations to posteriors; a decoder reconstructs observations — grounding latents in evidence.

In [4]:
class MiniRSSM(nn.Module):
    def __init__(self, obs_dim, action_dim, latent_dim):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Linear(obs_dim + action_dim, 128), nn.ReLU(), nn.Linear(128, 2 * latent_dim),
        )
        self.prior = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(), nn.Linear(128, 2 * latent_dim),
        )
        self.deterministic = nn.GRUCell(latent_dim + action_dim, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(2 * latent_dim, 128), nn.ReLU(), nn.Linear(128, obs_dim),
        )

    def imagine_step(self, h, z, a):
        h = self.deterministic(torch.cat([z, a], dim=-1), h)
        stats = self.prior(h)
        mu, logvar = stats.chunk(2, dim=-1)
        z = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
        obs = self.decoder(torch.cat([h, z], dim=-1))
        return h, z, obs

obs_dim, action_dim, latent_dim = 4, 1, 8
rssm = MiniRSSM(obs_dim, action_dim, latent_dim)
obs_batch = torch.tensor([t[0] for t in transitions[:32]], dtype=torch.float32)
act_batch = torch.tensor([[t[1]] for t in transitions[:32]], dtype=torch.float32)
enc = rssm.encoder(torch.cat([obs_batch, act_batch], dim=-1))
mu, logvar = enc.chunk(2, dim=-1)
z = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
h = torch.zeros(32, latent_dim)
recon = rssm.decoder(torch.cat([h, z], dim=-1))
loss = F.mse_loss(recon, obs_batch) + 0.01 * (-0.5 * (1 + logvar - mu.pow(2) - logvar.exp())).mean()
print(f"ELBO-style loss on mini-batch: {loss.item():.4f}")

ELBO-style loss on mini-batch: 0.2999


/tmp/ipykernel_3738/2465642595.py:26: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  obs_batch = torch.tensor([t[0] for t in transitions[:32]], dtype=torch.float32)


### 🛠️ B3. LeWorldModel Training Procedure

LeWorldModel's training uses two main loss terms: a prediction loss for next-embedding accuracy and a SIGReg regularization term to prevent representation collapse, ensuring stable and robust learning.

Here's a conceptual representation of the LeWorldModel training procedure, highlighting the prediction loss and SIGReg regularization:

In [5]:
import torch.nn.functional as F

# Placeholder functions for encoder, predictor, and SIGReg
# In a real implementation, these would be neural networks.

def encoder(obs_sequence):
    # Simulate encoding observations into latent embeddings
    # (B, T, C, H, W) -> (B, T, D)
    B, T, C, H, W = obs_sequence.shape
    D = 32 # Example latent dimension
    return torch.randn(B, T, D) # Returns random embeddings for conceptual demo

def predictor(embeddings, actions):
    # Simulate predicting the next-step embedding
    # (B, T, D), (B, T, A) -> (B, T, D)
    B, T, D = embeddings.shape
    return torch.randn(B, T, D) # Returns random predictions for conceptual demo

def SIGReg(embeddings_transposed):
    # Simulate SIGReg regularization to prevent collapse
    # (T, B, D) -> scalar loss term
    # For this conceptual demo, we'll return a simple mean of squared embeddings
    return (embeddings_transposed**2).mean()


def LeWorldModel_conceptual_training_loss(obs, actions, lambd=0.1):
    """
    obs: (B, T, C, H, W) raw pixels sequence
    actions: (B, T, A) action sequence
    lambd: (float) SIGReg loss weight
    """

    # Ensure obs and actions have the expected conceptual shape for the encoder/predictor
    # For a real implementation, 'obs' would likely be batched and processed
    # For this conceptual example, let's assume obs is (B, T, C, H, W) and actions is (B, T, A)
    # We need to simulate these shapes for the placeholder functions.
    B, T = obs.shape[0], obs.shape[1]

    emb = encoder(obs)  # (B, T, D)
    next_emb = predictor(emb, actions)  # (B, T, D)

    # -- LeWorldModel training loss
    # next-embedding prediction loss
    # Note: F.mse_loss expects (input, target), here emb[:, 1:] are targets for next_emb[:, :-1]
    # The original pseudocode (emb[:, 1:] - next_emb[:, :-1]) implies a direct difference.
    # Using mse_loss for conceptual clarity.
    pred_loss = F.mse_loss(next_emb[:, :-1], emb[:, 1:])

    # step-wise sigreg (anti-collapse)
    sigreg_loss = SIGReg(emb.transpose(0, 1))

    return pred_loss + lambd * sigreg_loss

# Example usage with dummy data (conceptual)
# Assuming a batch size of 2, time steps of 5, 3 channels, 64x64 pixels, 1 action dim
B, T, C, H, W, A = 2, 5, 3, 64, 64, 1
dummy_obs = torch.randn(B, T, C, H, W)
dummy_actions = torch.randn(B, T, A)

conceptual_total_loss = LeWorldModel_conceptual_training_loss(dummy_obs, dummy_actions)
print(f"Conceptual LeWorldModel Total Loss: {conceptual_total_loss.item():.4f}")

Conceptual LeWorldModel Total Loss: 2.1675


### 🛠️ B4. Policy gradient on imagined rollouts

 In **Dreamer**, a policy is trained on **imagined** trajectories from the world model using REINFORCE-style gradients — learning control without additional environment interaction.

In [6]:
class LinearPolicy(nn.Module):
    def __init__(self, dim, n_actions=2):
        super().__init__()
        self.fc = nn.Linear(dim, n_actions)

    def forward(self, h):
        logits = self.fc(h)
        return torch.distributions.Categorical(logits=logits)

policy = LinearPolicy(rssm.latent_dim * 2)
opt = torch.optim.Adam(policy.parameters(), lr=1e-2)
for step in range(50):
    h = torch.zeros(1, rssm.latent_dim)
    z = torch.randn(1, rssm.latent_dim)
    log_probs, rewards = [], []
    for _ in range(10):
        state = torch.cat([h, z], dim=-1)
        dist = policy(state)
        a = dist.sample()
        log_probs.append(dist.log_prob(a))
        a_cont = a.float().unsqueeze(-1)
        h, z, _ = rssm.imagine_step(h, z, a_cont)
        rewards.append(torch.randn(1))
    R = torch.stack(rewards).sum()
    loss = -R * torch.stack(log_probs).sum()
    opt.zero_grad()
    loss.backward()
    opt.step()
print(f"Policy training loss: {loss.item():.4f}")

Policy training loss: 54.1249
